# 11｜Choice实时行情与OpenBB查询验收

本Notebook使用Choice官方一次性行情快照`csqsnapshot`，仅发起一次三证券请求；随后把标准化结果写入独立SQLite快照表，并分别验证OpenBB `provider="choice"`和`provider="qianji"`。

运行前请应用0.10.0补丁、从头运行更新后的00号环境构建Notebook并彻底重启内核。Notebook不会打印账号、密码或Token。

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT_OVERRIDE = ""  # Notebook位于项目notebooks目录时保持为空

def find_project_root(start: Path) -> Path:
    if PROJECT_ROOT_OVERRIDE.strip():
        candidate = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        if (candidate / "src" / "qianji_data_mini").exists():
            return candidate
        raise FileNotFoundError(f"项目根目录不正确：{candidate}")
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "qianji_data_mini").exists():
            return candidate
    raise FileNotFoundError("未找到项目根目录，请填写PROJECT_ROOT_OVERRIDE。")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
OUTPUT_DIR = PROJECT_ROOT / "validation_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Python路径：", sys.executable)
print("项目根目录：", PROJECT_ROOT)
print("输出目录：", OUTPUT_DIR)

Python路径： d:\minicoda3\envs\dm311\python.exe
项目根目录： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini
输出目录： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\validation_output


In [2]:
from importlib.metadata import PackageNotFoundError, version
from packaging.version import Version
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env", override=True)

def installed_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "0.0.0"

qianji_version = installed_version("qianji-data-mini")
choice_version = installed_version("openbb-choice")
openbb_version = installed_version("openbb")
version_ok = Version(qianji_version) >= Version("0.10.0") and Version(choice_version) >= Version("0.2.0")
print("qianji-data-mini：", qianji_version)
print("openbb-choice：", choice_version)
print("OpenBB：", openbb_version)
print("Choice登录模式：", os.getenv("CHOICE_LOGIN_MODE", "auto"))
if not version_ok:
    raise RuntimeError("版本低于0.10.0/0.2.0，请运行更新后的00号Notebook并重启内核。")

qianji-data-mini： 0.10.0
openbb-choice： 0.2.0
OpenBB： 4.7.2
Choice登录模式： userinfo


In [3]:
SYMBOLS = ["000001.SZ", "600519.SH", "300750.SZ"]
CALL_CHOICE = True
STRICT_MODE = False

print("实时行情样本：", SYMBOLS)
print("本次Choice快照调用次数上限：1")

实时行情样本： ['000001.SZ', '600519.SH', '300750.SZ']
本次Choice快照调用次数上限：1


In [4]:
import pandas as pd
from IPython.display import display
from qianji_data_mini import Database, QuoteSnapshot

database = Database()
DB_PATH = database.path
with database.connect() as connection:
    sqlite_integrity = str(connection.execute("PRAGMA quick_check").fetchone()[0])
    quote_rows_before = int(connection.execute("SELECT COUNT(*) FROM equity_quote_snapshot").fetchone()[0])
    quote_runs_before = int(connection.execute("SELECT COUNT(*) FROM quote_ingestion_run").fetchone()[0])

print("SQLite数据库：", DB_PATH)
print("数据库完整性：", sqlite_integrity)
print("运行前快照行数：", quote_rows_before)
print("运行前快照采集记录：", quote_runs_before)

SQLite数据库： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\data\qianji_market.db
数据库完整性： ok
运行前快照行数： 0
运行前快照采集记录： 2


In [5]:
from openbb import obb

username = os.getenv("CHOICE_USERNAME", "")
password = os.getenv("CHOICE_PASSWORD", "")
login_mode = os.getenv("CHOICE_LOGIN_MODE", "auto").strip().lower()
if login_mode == "userinfo" or (login_mode == "auto" and not username and not password):
    os.environ["CHOICE_LOGIN_MODE"] = "userinfo"
    obb.user.credentials.choice_username = "local_userinfo"
    obb.user.credentials.choice_password = "local_userinfo"
else:
    obb.user.credentials.choice_username = username
    obb.user.credentials.choice_password = password

provider_routes = dict(getattr(obb.coverage, "providers", {}))
choice_routes = set(provider_routes.get("choice", []))
qianji_routes = set(provider_routes.get("qianji", []))
quote_route = ".equity.price.quote"
routes_ok = quote_route in choice_routes and quote_route in qianji_routes
print("Choice实时行情路由：", quote_route in choice_routes)
print("qianji实时行情路由：", quote_route in qianji_routes)
if not routes_ok:
    raise RuntimeError("OpenBB尚未发现实时行情路由，请重新运行00号Notebook并彻底重启内核。")

Choice实时行情路由： True
qianji实时行情路由： True


In [6]:
choice_error = ""
choice_provider = ""
choice_quotes = pd.DataFrame()
choice_call_attempted = False

if CALL_CHOICE:
    choice_call_attempted = True
    try:
        choice_result = obb.equity.price.quote(
            symbol=",".join(SYMBOLS), provider="choice", use_cache=False
        )
        choice_provider = choice_result.provider
        choice_quotes = choice_result.to_dataframe().reset_index(drop=True)
    except Exception as exc:
        choice_error = f"{type(exc).__name__}: {str(exc)[:1500]}"

print("Choice调用是否发起：", choice_call_attempted)
print("Choice返回行数：", len(choice_quotes))
print("Choice错误：", choice_error or "无")
display(choice_quotes)

[EmQuantAPI Python] [Em_Info][2026-09-02 18:25:54]:The current version is EmQuantAPI(V2.7.5.0).

[EmQuantAPI Python] [Em_Info][2026-09-02 18:25:54]:verifying your token...

[EmQuantAPI Python] [Em_Info][2026-09-02 18:25:54]:connect server...

[EmQuantAPI Python] [Em_Info][2026-09-02 18:25:55]:token login start success!

[EmQuantAPI Python] [Em_Info][2026-09-02 18:25:56]:updating ChoiceToHQ.xml from version 0 to 120

[EmQuantAPI Python] [Em_Info][2026-09-02 18:25:56]:loading ChoiceToHQ.xml...

[EmQuantAPI Python] [Em_Info][2026-09-02 18:25:56]:DownLoad D:/EMQuantAPI_Python/python3/libs/windows/bjse_code_conversion.txt success.

[EmQuantAPI Python] [Em_Error][2026-09-02 18:25:56]:[csq] fail: [10001012] insufficient user access

[EmQuantAPI Python] [Em_Info][2026-09-02 18:25:56]:heartbeatthread end.

Choice调用是否发起： True
Choice返回行数： 0
Choice错误： OpenBBError: 
[Error] -> Choice csqsnapshot failed (ErrorCode=10001012): insufficient user access


""


In [7]:
from datetime import datetime, timezone

def optional_number(value):
    return None if value is None or pd.isna(value) else float(value)

quote_records = []
for _, row in choice_quotes.iterrows():
    quote_time = pd.to_datetime(row.get("quote_time") or row.get("last_timestamp")).to_pydatetime()
    fetched_at = pd.to_datetime(row.get("fetched_at") or datetime.now(timezone.utc)).to_pydatetime()
    quote_records.append(QuoteSnapshot(
        symbol=str(row["symbol"]), quote_time=quote_time,
        open=optional_number(row.get("open")), high=optional_number(row.get("high")),
        low=optional_number(row.get("low")), last_price=optional_number(row.get("last_price")),
        previous_close=optional_number(row.get("prev_close")),
        volume=optional_number(row.get("volume")), amount=optional_number(row.get("amount")),
        source="choice", fetched_at=fetched_at,
        raw={"openbb_provider": "choice", "snapshot_method": "csqsnapshot"},
    ))

stored_rows = database.upsert_quote_snapshots(quote_records)
with database.connect() as connection:
    quote_rows_after_first = int(connection.execute("SELECT COUNT(*) FROM equity_quote_snapshot").fetchone()[0])
database.upsert_quote_snapshots(quote_records)
with database.connect() as connection:
    quote_rows_after_second = int(connection.execute("SELECT COUNT(*) FROM equity_quote_snapshot").fetchone()[0])

finished_at = datetime.now(timezone.utc)
database.log_quote_ingestion(
    source="choice", requested_symbols=SYMBOLS, received_rows=len(quote_records),
    stored_rows=stored_rows, errors=({"choice_quote": choice_error} if choice_error else {}),
    started_at=finished_at.isoformat(), finished_at=finished_at.isoformat(),
)
print("标准化/写入行数：", len(quote_records), stored_rows)
print("第一次后/重复写入后总行数：", quote_rows_after_first, quote_rows_after_second)

标准化/写入行数： 0 0
第一次后/重复写入后总行数： 0 0


In [8]:
qianji_error = ""
qianji_provider = ""
qianji_quotes = pd.DataFrame()
if quote_records:
    try:
        qianji_result = obb.equity.price.quote(
            symbol=",".join(SYMBOLS), source="choice",
            provider="qianji", use_cache=False,
        )
        qianji_provider = qianji_result.provider
        qianji_quotes = qianji_result.to_dataframe().reset_index(drop=True)
    except Exception as exc:
        qianji_error = f"{type(exc).__name__}: {str(exc)[:1500]}"

sqlite_latest = database.query_quote_snapshots(
    symbols=SYMBOLS, source="choice", latest_only=True
)
print("qianji返回行数：", len(qianji_quotes))
print("qianji错误：", qianji_error or "无")
display(sqlite_latest)
display(qianji_quotes)

qianji返回行数： 0
qianji错误： 无


,source,symbol,quote_time,open,high,low,last_price,previous_close,volume,amount,currency,timezone,volume_unit,amount_unit,fetched_at


""


In [9]:
import math

FIELD_MAP = {
    "open": ("open", "open"), "high": ("high", "high"),
    "low": ("low", "low"), "last_price": ("last_price", "last_price"),
    "prev_close": ("previous_close", "prev_close"),
    "volume": ("volume", "volume"), "amount": ("amount", "amount"),
}

def frame_row(frame, symbol):
    matched = frame[frame["symbol"].astype(str) == symbol] if not frame.empty and "symbol" in frame else pd.DataFrame()
    return None if matched.empty else matched.iloc[0]

def numbers_match(left, right):
    if left is None or pd.isna(left):
        return right is None or pd.isna(right)
    if right is None or pd.isna(right):
        return False
    return math.isclose(float(left), float(right), rel_tol=1e-10, abs_tol=1e-8)

reconciliation_rows = []
for symbol in SYMBOLS:
    choice_row = frame_row(choice_quotes, symbol)
    sqlite_row = frame_row(sqlite_latest, symbol)
    qianji_row = frame_row(qianji_quotes, symbol)
    for field, (sqlite_field, qianji_field) in FIELD_MAP.items():
        choice_value = None if choice_row is None else choice_row.get(field)
        sqlite_value = None if sqlite_row is None else sqlite_row.get(sqlite_field)
        qianji_value = None if qianji_row is None else qianji_row.get(qianji_field)
        reconciliation_rows.append({
            "symbol": symbol, "field": field, "choice_value": choice_value,
            "sqlite_value": sqlite_value, "qianji_value": qianji_value,
            "choice_sqlite_match": numbers_match(choice_value, sqlite_value),
            "sqlite_qianji_match": numbers_match(sqlite_value, qianji_value),
        })
reconciliation = pd.DataFrame(reconciliation_rows)
display(reconciliation)

,symbol,field,choice_value,sqlite_value,qianji_value,choice_sqlite_match,sqlite_qianji_match
0,000001.SZ,open,None,None,None,True,True
1,000001.SZ,high,None,None,None,True,True
2,000001.SZ,low,None,None,None,True,True
3,000001.SZ,last_price,None,None,None,True,True
4,000001.SZ,prev_close,None,None,None,True,True
5,000001.SZ,volume,None,None,None,True,True
6,000001.SZ,amount,None,None,None,True,True
7,600519.SH,open,None,None,None,True,True
8,600519.SH,high,None,None,None,True,True
9,600519.SH,low,None,None,None,True,True


In [10]:
with database.connect() as connection:
    quote_runs_after = int(connection.execute("SELECT COUNT(*) FROM quote_ingestion_run").fetchone()[0])

def ohlc_valid(row):
    values = [row.get(name) for name in ("open", "high", "low", "last_price")]
    numeric = [float(value) for value in values if value is not None and not pd.isna(value) and float(value) > 0]
    if not numeric:
        return False
    high, low = row.get("high"), row.get("low")
    return not (high is not None and low is not None) or float(high) >= float(low)

quality_rows = []
def gate(name, passed, evidence):
    quality_rows.append({"check": name, "status": "PASS" if passed else "FAIL", "evidence": str(evidence)})

choice_symbols = set(choice_quotes.get("symbol", pd.Series(dtype=str)).astype(str))
qianji_symbols = set(qianji_quotes.get("symbol", pd.Series(dtype=str)).astype(str))
quote_times_ok = not choice_quotes.empty and choice_quotes.get("quote_time", pd.Series(dtype=object)).notna().all()
ohlc_ok = not choice_quotes.empty and all(ohlc_valid(row) for _, row in choice_quotes.iterrows())
units_ok = not choice_quotes.empty and choice_quotes.get("volume_unit", pd.Series(dtype=str)).eq("share").all() and choice_quotes.get("amount_unit", pd.Series(dtype=str)).eq("CNY").all()
match_ok = not reconciliation.empty and reconciliation[["choice_sqlite_match", "sqlite_qianji_match"]].all().all()

gate("插件版本达到0.10.0/0.2.0", version_ok, f"{qianji_version}/{choice_version}")
gate("SQLite完整性", sqlite_integrity == "ok", sqlite_integrity)
gate("Choice与qianji实时行情路由已注册", routes_ok, quote_route)
gate("Choice一次快照调用无错误", CALL_CHOICE and not choice_error, choice_error or "one bounded call")
gate("Choice返回全部样本证券", choice_symbols == set(SYMBOLS), sorted(choice_symbols))
gate("Choice结果Provider正确", choice_provider == "choice", choice_provider)
gate("行情时间存在", quote_times_ok, choice_quotes.get("quote_time", pd.Series(dtype=object)).tolist())
gate("价格区间基本有效", ohlc_ok, f"rows={len(choice_quotes)}")
gate("成交量与成交额单位明确", units_ok, "share/CNY")
gate("快照成功落库", set(sqlite_latest.get("symbol", pd.Series(dtype=str))) >= set(SYMBOLS), len(sqlite_latest))
gate("重复写入不增加行数", quote_rows_after_first == quote_rows_after_second, f"{quote_rows_after_first}->{quote_rows_after_second}")
gate("qianji返回全部样本证券", qianji_symbols == set(SYMBOLS) and not qianji_error, sorted(qianji_symbols))
gate("qianji结果Provider正确", qianji_provider == "qianji", qianji_provider)
gate("Choice-SQLite-qianji数值一致", match_ok, f"rows={len(reconciliation)}")
gate("本次新增一条采集审计记录", quote_runs_after == quote_runs_before + 1, f"{quote_runs_before}->{quote_runs_after}")

quality_gates = pd.DataFrame(quality_rows)
failed_count = int((quality_gates["status"] == "FAIL").sum())
display(quality_gates)
print("通过：", len(quality_gates) - failed_count, "失败：", failed_count)

,check,status,evidence
0,插件版本达到0.10.0/0.2.0,PASS,0.10.0/0.2.0
1,SQLite完整性,PASS,ok
2,Choice与qianji实时行情路由已注册,PASS,.equity.price.quote
3,Choice一次快照调用无错误,FAIL,OpenBBError: \n[Error] -> Choice csqsnapshot f...
4,Choice返回全部样本证券,FAIL,[]
5,Choice结果Provider正确,FAIL,
6,行情时间存在,FAIL,[]
7,价格区间基本有效,FAIL,rows=0
8,成交量与成交额单位明确,FAIL,share/CNY
9,快照成功落库,FAIL,0


通过： 6 失败： 9


In [11]:
data_map = pd.DataFrame([
    {"stage": "vendor", "component": "Choice EmQuantAPI", "interface": "csqsnapshot", "provider": "choice"},
    {"stage": "storage", "component": "SQLite", "interface": "equity_quote_snapshot", "provider": ""},
    {"stage": "consumption", "component": "OpenBB", "interface": "obb.equity.price.quote", "provider": "qianji"},
])
display(data_map)

,stage,component,interface,provider
0,vendor,Choice EmQuantAPI,csqsnapshot,choice
1,storage,SQLite,equity_quote_snapshot,
2,consumption,OpenBB,obb.equity.price.quote,qianji


In [12]:
import json
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill

timestamp = datetime.now().astimezone().strftime("%Y%m%d_%H%M%S")
excel_path = OUTPUT_DIR / f"Choice实时行情_OpenBB查询验收_{timestamp}.xlsx"
json_path = OUTPUT_DIR / f"Choice实时行情_OpenBB查询验收_{timestamp}.json"
overview = pd.DataFrame([
    ["generated_at", datetime.now(timezone.utc).isoformat()], ["python", sys.executable],
    ["database", str(DB_PATH)], ["qianji_data_mini_version", qianji_version],
    ["openbb_choice_version", choice_version], ["openbb_version", openbb_version],
    ["choice_snapshot_calls", 1 if choice_call_attempted else 0],
    ["passed", len(quality_gates) - failed_count], ["failed", failed_count],
], columns=["item", "value"])
errors = pd.DataFrame([{"stage": "choice", "error": choice_error}, {"stage": "qianji", "error": qianji_error}])
sheets = {"验收概览": overview, "质量门槛": quality_gates, "错误记录": errors, "Choice直接结果": choice_quotes, "SQLite最新快照": sqlite_latest, "qianji查询结果": qianji_quotes, "逐字段核对": reconciliation, "数据地图": data_map}
def excel_safe(frame):
    safe = frame.copy()
    for column in safe.columns:
        if isinstance(safe[column].dtype, pd.DatetimeTZDtype):
            safe[column] = safe[column].map(lambda value: value.isoformat() if pd.notna(value) else None)
        elif safe[column].dtype == object:
            safe[column] = safe[column].map(lambda value: value.isoformat() if isinstance(value, datetime) else value)
    return safe
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    for sheet_name, frame in sheets.items():
        excel_safe(frame).to_excel(writer, sheet_name=sheet_name[:31], index=False)
workbook = load_workbook(excel_path)
for worksheet in workbook.worksheets:
    for cell in worksheet[1]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = PatternFill("solid", fgColor="2F75B5")
        cell.alignment = Alignment(horizontal="center")
    worksheet.freeze_panes = "A2"
    worksheet.auto_filter.ref = worksheet.dimensions
    for column in worksheet.columns:
        width = min(45, max(10, max(len(str(cell.value or "")) for cell in column) + 2))
        worksheet.column_dimensions[column[0].column_letter].width = width
workbook.save(excel_path)

payload = {"generated_at": datetime.now(timezone.utc).isoformat(), "database": str(DB_PATH), "qianji_version": qianji_version, "choice_version": choice_version, "choice_snapshot_calls": 1 if choice_call_attempted else 0, "quality_gates": quality_gates.to_dict("records"), "errors": {"choice": choice_error, "qianji": qianji_error}}
json_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print("Excel证据：", excel_path)
print("JSON证据：", json_path)

Excel证据： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\validation_output\Choice实时行情_OpenBB查询验收_20260902_182557.xlsx
JSON证据： D:\OneDrive\桌面\数据基座代码\qianji_openbb_mini\validation_output\Choice实时行情_OpenBB查询验收_20260902_182557.json


In [13]:
if failed_count == 0:
    print("✅ 11号验收通过：Choice实时行情已完成一次性获取、SQLite落库和OpenBB双Provider查询。")
else:
    print(f"⚠️ 11号验收存在{failed_count}项失败，请查看质量门槛与错误记录。")
    display(quality_gates[quality_gates["status"] == "FAIL"])
if STRICT_MODE and failed_count:
    raise RuntimeError(f"11号验收存在{failed_count}项失败。")

⚠️ 11号验收存在9项失败，请查看质量门槛与错误记录。


,check,status,evidence
3,Choice一次快照调用无错误,FAIL,OpenBBError: \n[Error] -> Choice csqsnapshot f...
4,Choice返回全部样本证券,FAIL,[]
5,Choice结果Provider正确,FAIL,
6,行情时间存在,FAIL,[]
7,价格区间基本有效,FAIL,rows=0
8,成交量与成交额单位明确,FAIL,share/CNY
9,快照成功落库,FAIL,0
11,qianji返回全部样本证券,FAIL,[]
12,qianji结果Provider正确,FAIL,
